In [2]:
from statsmodels.regression.rolling import RollingOLS
import pandas_datareader.data as web
import matplotlib.pyplot as plt
import statsmodels.api as sm
import pandas as pd
import numpy as np
from datetime import datetime
import yfinance as yf
from datetime import datetime
# import pandas_ta
import warnings
warnings.filterwarnings('ignore')


In [33]:
#Download/Load SP500 stocks prices data.
sp500 = pd.read_html('https://en.wikipedia.org/wiki/List_of_S%26P_500_companies')[0]

sp500['Symbol'] = sp500['Symbol'].str.replace('.', '-')

symbols_list = sp500['Symbol'].unique().tolist()

end_date = '2025-02-27'

start_date = pd.to_datetime(end_date)-pd.DateOffset(365*2)

df = yf.download(tickers=symbols_list,
                 start=start_date,
                 end=end_date, auto_adjust=False).stack()

df.index.names = ['date', 'ticker']

df.columns = df.columns.str.lower()

df

YF.download() has changed argument auto_adjust default to True


[*********************100%***********************]  503 of 503 completed


Price                   close        high         low        open      volume
date       ticker                                                            
2023-02-28 A       139.974670  141.364857  139.402820  139.757761   2342500.0
           AAPL    145.938248  147.591572  145.364037  145.581841  50547000.0
           ABBV    142.718216  143.154069  141.531217  142.773868   7561700.0
           ABNB    123.279999  126.384003  123.029999  125.110001   7317100.0
           ABT      97.703789   98.376148   95.753940   95.898012   7877000.0
...                       ...         ...         ...         ...         ...
2025-02-26 XYL     128.500000  129.570007  128.199997  128.669998   1280800.0
           YUM     151.500000  155.179993  151.169998  154.899994   2112000.0
           ZBH     104.919998  107.580002  103.940002  106.800003   1719400.0
           ZBRA    319.480011  323.690002  312.929993  314.869995    572200.0
           ZTS     164.990005  169.000000  164.880005  168.210007   2609700.0

[250920 rows x 5 columns]

In [44]:
# Filter the stacked DataFrame for 'PLTR'
pltr_data = df.xs('PLTR', level='ticker')

# Calculate the one-year return for 'PLTR' using the 'adj close' column
one_year_return = float(pltr_data['close'].pct_change(periods=252).iloc[-1] * 100)

print(f"PLTR One-Year Return: {one_year_return:.2f}%")

PLTR One-Year Return: 288.81%


In [45]:
# Create an empty list to store the results
results = []

# Get the list of all tickers in the stacked dataframe
tickers = df.index.get_level_values('ticker').unique()

# Iterate over each ticker and calculate the current price and one-year return
for ticker in tickers:
    ticker_data = df.xs(ticker, level='ticker')
    try:
        current_price = float(ticker_data['close'].iloc[-1])
        one_year_return = float(ticker_data['close'].pct_change(periods=252).iloc[-1] * 100)
        results.append({
            'Ticker': ticker, 
            'Current Price': current_price, 
            'One Year Return (%)': one_year_return
        })
    except Exception as e:
        # Handle any potential errors, such as insufficient data
        print(f"Error processing {ticker}: {e}")

# Create a DataFrame from the results
stocks_df = pd.DataFrame(results)

# Display the DataFrame
print(stocks_df)

    Ticker  Current Price  One Year Return (%)
0        A     134.470001             2.554761
1     AAPL     240.360001            32.311499
2     ABBV     203.009995            18.145407
3     ABNB     144.029999            -5.653088
4      ABT     135.960007            16.131549
..     ...            ...                  ...
498   KVUE      23.219999            25.846702
499   VLTO      99.720001            15.444098
500    GEV     335.239990                  NaN
501   SOLV      84.040001                  NaN
502     SW      53.520000                  NaN

[503 rows x 3 columns]


In [46]:
stocks_df = stocks_df.sort_values(by='One Year Return (%)', ascending=False)
print(stocks_df.head())

    Ticker  Current Price  One Year Return (%)
373   PLTR      89.309998           288.811494
470    VST     148.190002           207.867416
439    TPL    1351.469971           164.980666
338    NRG     113.330002           124.241798
453    UAL      97.400002           114.018908


In [47]:
# Identify tickers with NaN values in the "One Year Return (%)" column
nan_tickers = stocks_df[stocks_df['One Year Return (%)'].isna()]['Ticker'].tolist()

# Print the tickers with NaN returns
print("Tickers with NaN in One Year Return:")
print(nan_tickers)

Tickers with NaN in One Year Return:
['GEV', 'SOLV', 'SW']
